**Rules for the code:**

- Include all the code you used for your report in this file. The code for any section in the report should go under the same section in this file.
- Any missing code will result in -20% from its corresponding section in the report.
- Any irrelevant code will result in -20% from its corresponding section in the report.
- Make sure that you run your code before rendering so that all the necessary visual/numeric outputs are visible.
- Any code that is not properly run or throws errors will be considered missing/irrelevant.

## 4) Data

In [5]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.formula.api as smf
from sklearn.linear_model import LinearRegression, RidgeCV, LassoCV, LogisticRegression
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.metrics import root_mean_squared_error, r2_score, confusion_matrix, classification_report

# test train split
from sklearn.model_selection import train_test_split

In [6]:
data = pd.read_csv('uber.csv')
data = data.drop(columns=['Unnamed: 0', 'key'])
data.dropna(inplace=True)

data['pickup_datetime'] = pd.to_datetime(data['pickup_datetime'])
data['month'] = data['pickup_datetime'].dt.month
data['day'] = data['pickup_datetime'].dt.day
data['hour'] = data['pickup_datetime'].dt.hour
data['day_of_week'] = data['pickup_datetime'].dt.dayofweek

data.drop(columns=['pickup_datetime'], inplace=True)

def haversine(lat1, lon1, lat2, lon2):
    R = 6371  # Earth radius in kilometers

    lat1 = np.radians(lat1)
    lon1 = np.radians(lon1)
    lat2 = np.radians(lat2)
    lon2 = np.radians(lon2)

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = np.sin(dlat / 2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2)**2
    c = 2 * np.arctan2(np.sqrt(a), np.sqrt(1 - a))

    return R * c
data['distance'] = haversine(data['pickup_latitude'], data['pickup_longitude'], data['dropoff_latitude'], data['dropoff_longitude'])

data = data[(data["fare_amount"] > 0) & (data["fare_amount"] < 200)]
data = data[(data["distance"] > 0) & (data["distance"] < 100)]
data = data[(data["passenger_count"] > 0) & (data["passenger_count"] < 7)]

## 5) Prediction

In [7]:
X_train, X_test, y_train, y_test = train_test_split(data[["distance"]], data["fare_amount"], test_size=0.2, random_state=1)
train = pd.concat([X_train, y_train], axis=1)
test = pd.concat([X_test, y_test], axis=1)
model = smf.ols(formula='fare_amount ~ distance', data=train).fit()
model.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:            fare_amount   R-squared:                       0.715
Model:                            OLS   Adj. R-squared:                  0.714
Method:                 Least Squares   F-statistic:                 3.868e+05
Date:                Sun, 15 Mar 2026   Prob (F-statistic):               0.00
Time:                        13:37:25   Log-Likelihood:            -4.7293e+05
No. Observations:              154550   AIC:                         9.459e+05
Df Residuals:                  154548   BIC:                         9.459e+05
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
==============================================================================
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept      3.9964      0.018    226.178      0.000       3.962       4.031
distance       2.1860      0.004    621.915      0.000       2.179       2.193
==============================================================================
Omnibus:                    86867.235   Durbin-Watson:                   1.997
Prob(Omnibus):                  0.000   Jarque-Bera (JB):        815884567.895
Skew:                          -0.814   Prob(JB):                         0.00
Kurtosis:                     358.943   Cond. No.                         6.89
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
"""

In [8]:
predictors = ['distance', 'pickup_latitude', 'pickup_longitude', 'dropoff_latitude', 'dropoff_longitude', 'passenger_count', 'hour', 'day', 'month', 'day_of_week']

results = []

for i in range(len(predictors)):
    X_train, X_test, y_train, y_test = train_test_split(data[predictors[:i+1]], data["fare_amount"], test_size=0.3, random_state=1)
    model = LinearRegression()
    model.fit(X_train, y_train)
    train_pred = model.predict(X_train)
    test_pred = model.predict(X_test)
    train_rmse = root_mean_squared_error(y_train, train_pred)
    test_rmse = root_mean_squared_error(y_test, test_pred)
    results.append((predictors[:i+1], train_rmse, test_rmse))

model_performances = pd.DataFrame(results, columns=["Predictors", "Train RMSE", "Test RMSE"])
pd.set_option('display.max_colwidth', None)
model_performances

,Predictors,Train RMSE,Test RMSE
0,[distance],5.007371,5.356812
1,"[distance, pickup_latitude]",5.005330,5.357931
2,"[distance, pickup_latitude, pickup_longitude]",5.003474,5.357818
3,"[distance, pickup_latitude, pickup_longitude, dropoff_latitude]",5.002981,5.357742
4,"[distance, pickup_latitude, pickup_longitude, dropoff_latitude, dropoff_longitude]",4.993727,5.331473
5,"[distance, pickup_latitude, pickup_longitude, dropoff_latitude, dropoff_longitude, passenger_count]",4.993206,5.331662
6,"[distance, pickup_latitude, pickup_longitude, dropoff_latitude, dropoff_longitude, passenger_count, hour]",4.993093,5.331708
7,"[distance, pickup_latitude, pickup_longitude, dropoff_latitude, dropoff_longitude, passenger_count, hour, day]",4.993082,5.331733
8,"[distance, pickup_latitude, pickup_longitude, dropoff_latitude, dropoff_longitude, passenger_count, hour, day, month]",4.990941,5.330346
9,"[distance, pickup_latitude, pickup_longitude, dropoff_latitude, dropoff_longitude, passenger_count, hour, day, month, day_of_week]",4.990535,5.329623


In [9]:
X_train, X_test, y_train, y_test = train_test_split(data[predictors], data["fare_amount"], test_size=0.3, random_state=1)

poly = PolynomialFeatures(degree=3, include_bias=False)
X_train_poly = poly.fit_transform(X_train)
X_test_poly = poly.transform(X_test)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_poly)
X_test_scaled = scaler.transform(X_test_poly)

alphas = np.logspace(-3, 3, 200) 
ridge_CV = RidgeCV(alphas=alphas, cv=5, scoring='neg_root_mean_squared_error')
ridge_CV.fit(X_train_scaled, y_train)
best_alpha = ridge_CV.alpha_
train_pred = ridge_CV.predict(X_train_scaled)
test_pred = ridge_CV.predict(X_test_scaled)
train_rmse = root_mean_squared_error(y_train, train_pred)
test_rmse = root_mean_squared_error(y_test, test_pred)  
print(f"Best alpha: {best_alpha}")
print(f"Train RMSE: {train_rmse}")
print(f"Test RMSE: {test_rmse}")

Best alpha: 1.1895340673703196
Train RMSE: 4.540749949906305
Test RMSE: 4.629173157034097


In [10]:
feature_names = poly.get_feature_names_out()
coef_df = pd.DataFrame({"feature": feature_names,"coefficient": ridge_CV.coef_})
coef_df.sort_values(by="coefficient", key=abs,  ascending=False)[:10]

,feature,coefficient
0,distance,-33.172817
99,distance dropoff_longitude^2,31.594559
66,distance^2 pickup_latitude,-24.669595
86,distance pickup_longitude dropoff_longitude,21.274191
67,distance^2 pickup_longitude,-20.843400
13,distance dropoff_latitude,-17.625976
10,distance^2,-15.910831
18,distance month,-15.809674
11,distance pickup_latitude,-15.648572
69,distance^2 dropoff_longitude,-15.225647


In [11]:
coef_df.sort_values(by="coefficient", key=abs,  ascending=True)[:5]

,feature,coefficient
264,passenger_count day_of_week^2,0.006505
203,dropoff_latitude^2 passenger_count,0.007511
258,passenger_count hour day_of_week,0.008519
272,hour month^2,0.011227
279,day month day_of_week,-0.012661


## 6) Inference

In [12]:
model = smf.ols(formula='fare_amount ~ distance + pickup_latitude + pickup_longitude + dropoff_latitude + dropoff_longitude +' \
'distance*pickup_latitude + distance*pickup_longitude + distance*dropoff_latitude + distance*dropoff_longitude', data=data).fit()
model.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:            fare_amount   R-squared:                       0.757
Model:                            OLS   Adj. R-squared:                  0.757
Method:                 Least Squares   F-statistic:                 6.685e+04
Date:                Sun, 15 Mar 2026   Prob (F-statistic):               0.00
Time:                        13:39:51   Log-Likelihood:            -5.7529e+05
No. Observations:              193188   AIC:                         1.151e+06
Df Residuals:                  193178   BIC:                         1.151e+06
Df Model:                           9                                         
Covariance Type:            nonrobust                                         
==============================================================================================
                                 coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------------------------------------
Intercept                     75.9023      1.285     59.081      0.000      73.384      78.420
distance                     -73.9684      0.462   -160.132      0.000     -74.874     -73.063
pickup_latitude               -7.6454      0.531    -14.410      0.000      -8.685      -6.606
pickup_longitude              27.3837      0.417     65.660      0.000      26.566      28.201
dropoff_latitude               9.7909      0.532     18.393      0.000       8.748      10.834
dropoff_longitude            -25.2222      0.415    -60.739      0.000     -26.036     -24.408
distance:pickup_latitude       0.2129      0.015     13.979      0.000       0.183       0.243
distance:pickup_longitude     -2.2565      0.022   -103.585      0.000      -2.299      -2.214
distance:dropoff_latitude     -2.5109      0.023   -110.415      0.000      -2.555      -2.466
distance:dropoff_longitude    -0.0417      0.018     -2.329      0.020      -0.077      -0.007
==============================================================================
Omnibus:                   246577.101   Durbin-Watson:                   2.003
Prob(Omnibus):                  0.000   Jarque-Bera (JB):        183395800.206
Skew:                           6.528   Prob(JB):                         0.00
Kurtosis:                     153.376   Cond. No.                     7.25e+04
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
[2] The condition number is large, 7.25e+04. This might indicate that there are
strong multicollinearity or other numerical problems.
"""